# Multi-Seed Split Summary (v2 data)

**Run order:** after `scripts/multi_seed_v2.sh` has executed the datagen, Phase 2, Phase 3 and
the locked-test eval for each seed. Reads only saved artifacts; trains nothing.
**Produces:** `models/multi-seed/summary.json` and the tables below.

Why this exists (`docs/EVALUATION.md` §2, §4): the v2 split is grouped by sender domain, so which
large domains land in the held-out split decides the false-positive rate. One split is one draw.
Each seed here rebuilds the split (same sources, same config, different seed), retrains both
phases with the training seed held at 42, and rescores with the validation-chosen operating points.
The spread across seeds is the between-split uncertainty that no within-split bootstrap can show.


## 1. Collect the per-seed artifacts


In [ ]:
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

DATASET_VERSION = "v2"


def resolve_project_dir():
    candidates = [Path(p) for p in (os.environ.get("EMAIL_NN_DIR"), "/workspace/training/email-classifier-nn", "/home/spark/projects/training/email-classifier-nn") if p]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (parent / "docs" / "PLAN.md").exists() and (parent / "notebooks").is_dir():
            return parent
    raise FileNotFoundError("project root not found")


PROJECT_DIR = resolve_project_dir()
builds = {}
for data_dir in sorted((PROJECT_DIR / "data").glob(f"source-clean-{DATASET_VERSION}*")):
    manifest_path = data_dir / "manifest.json"
    if not manifest_path.exists():
        continue
    m = re.search(r"-seed(\d+)$", data_dir.name)
    seed = int(m.group(1)) if m else 42
    manifest = json.loads(manifest_path.read_text())
    fp = manifest["config_fingerprint"][:16]
    eval_path = PROJECT_DIR / "models" / f"eval-{DATASET_VERSION}-{fp}" / "eval_locked_test.json"
    if not eval_path.exists():
        print(f"seed {seed}: data built ({fp}) but no eval yet; skipped")
        continue
    builds[seed] = {"fingerprint": fp, "manifest": manifest, "eval": json.loads(eval_path.read_text()),
                    "phase3": json.loads((PROJECT_DIR / "models" / f"phase3-cnn-bigru-{DATASET_VERSION}-{fp}" / "metrics.json").read_text()),
                    "phase2": json.loads((PROJECT_DIR / "models" / f"phase2-{DATASET_VERSION}-{fp}" / "metrics.json").read_text())}
print("seeds with complete runs:", {s: b["fingerprint"] for s, b in sorted(builds.items())})
assert builds, "no completed seed runs found"


## 2. Per-seed table and across-seed range

Operating points are the ones the eval notebook chose on each seed's own validation split.


In [ ]:
rows = []
for seed, b in sorted(builds.items()):
    ev = b["eval"]["models"]
    for model in ("cnn_bigru", "xgboost"):
        f1max = ev[model]["operating_points"]["f1_max_on_val"]["test"]
        alert = ev[model]["operating_points"]["alert_fpr_1pct"]
        act = ev[model]["operating_points"]["act_fpr_0.1pct"]
        top = alert["test_top_fp_domains"][:3]
        rows.append({
            "seed": seed, "model": model, "fingerprint": b["fingerprint"],
            "test_rows": b["manifest"]["counts"]["test"]["rows"],
            "f1max_recall": f1max["recall"], "f1max_fpr": f1max["false_positive_rate"], "f1max_f1": f1max["f1"],
            "alert_threshold": alert["threshold"], "alert_recall": alert["test"]["recall"], "alert_fpr": alert["test"]["false_positive_rate"],
            "alert_fpr_domain_ci": f"{alert['test_ci_domains']['false_positive_rate']['low']:.4f}-{alert['test_ci_domains']['false_positive_rate']['high']:.4f}",
            "act_recall": act["test"]["recall"], "act_fpr": act["test"]["false_positive_rate"],
            "adversarial_recall_alert": alert["adversarial_recall"],
            "roc_auc": f1max["roc_auc"], "average_precision": f1max["average_precision"],
            "top_fp_domains_alert": "; ".join(f"{d['domain']} ({d['false_positives']}/{d['legit_rows']})" for d in top),
        })
per_seed = pd.DataFrame(rows)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 80)
print("PER SEED (test split of each build)")
print(per_seed.drop(columns=["fingerprint"]).to_string(index=False))

metrics = ["f1max_recall", "f1max_fpr", "f1max_f1", "alert_recall", "alert_fpr", "act_recall", "act_fpr", "adversarial_recall_alert", "roc_auc", "average_precision"]
spread = per_seed.groupby("model")[metrics].agg(["min", "median", "max"])
print()
print("ACROSS SEEDS (min / median / max)")
print(spread.T.to_string())


## 3. Save


In [ ]:
summary = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_version": DATASET_VERSION,
    "seeds": {str(s): {"fingerprint": b["fingerprint"], "test_rows": b["manifest"]["counts"]["test"]["rows"]} for s, b in sorted(builds.items())},
    "training_seed": 42,
    "per_seed": per_seed.to_dict(orient="records"),
    "across_seeds": {model: {metric: {stat: float(spread.loc[model, (metric, stat)]) for stat in ("min", "median", "max")} for metric in metrics} for model in spread.index},
    "note": "each seed rebuilds the grouped split with the same sources and config; operating points are chosen on that seed's validation split",
}
out = PROJECT_DIR / "models" / "multi-seed" / "summary.json"
out.parent.mkdir(parents=True, exist_ok=True)
tmp = Path(str(out) + ".tmp")
tmp.write_text(json.dumps(summary, indent=2, default=str))
os.replace(tmp, out)
print("written:", out)
for model in ("cnn_bigru", "xgboost"):
    a = summary["across_seeds"][model]
    print(f"{model}: alert-tier test FPR across seeds {a['alert_fpr']['min']:.4f} to {a['alert_fpr']['max']:.4f} (median {a['alert_fpr']['median']:.4f}); "
          f"alert-tier recall {a['alert_recall']['min']:.3f} to {a['alert_recall']['max']:.3f}; F1-max FPR {a['f1max_fpr']['min']:.4f} to {a['f1max_fpr']['max']:.4f}")
